# AG News: attention-head fault tolerance in an encoder-only Transformer

Replication of the `transformers_EXP_A` protocol on a **real** text corpus.

This notebook addresses limitation 2 of the source study - *"Experiment A used a controlled synthetic classification dataset. Future work should validate the classification findings using real-world datasets."*

**Dataset**: AG News, 120 000 training / 7 600 test news items, 4 balanced topic classes.

**Model**: the fixed *Base* configuration - 4 encoder layers x 8 heads = **32 attention heads**, `d_model` 256, `d_ff` 1024, dropout 0.05, word-level embedding, CLS-token classification. Architecture is identical in all three modality notebooks so that layer-position findings are comparable.

**Research questions** (as in the source study)

- **RQ1** Do Transformer models contain critical and redundant attention heads?
- **RQ2** Is degradation explained by the number of failed heads alone, or also by which heads fail?
- **RQ3** How do the fault-tolerance results compare across datasets and modalities, given one fixed architecture?

**Hypotheses**

- **H1** Attention-head criticality is non-uniform across the architecture.
- **H2** Layer-wise fragility is non-monotonic and task-dependent.
- **H3** Degradation increases with the number of failed attention heads.
- **H4** Criticality-aware fault severity adds information beyond the raw fault count.
- **H5** Configurations containing more critical heads are likelier to exceed an acceptable degradation threshold.

## Imports

In [1]:
import json, pathlib, sys, time

import numpy as np
import pandas as pd
import torch

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from faulttol import config as C
from faulttol import multihead, plots, saferegion, single_head
from faulttol.injection import collect_attention_stats_per_head, get_all_heads
from faulttol.runtime import configure_threads, device_report, select_device
from faulttol.training import make_loader

print("torch", torch.__version__, "| threads", configure_threads(),
      "| cuda build", torch.version.cuda, "| cuda available", torch.cuda.is_available())
print("root:", ROOT)

torch 2.14.0+cu126 | threads 8 | cuda build 12.6 | cuda available True
root: D:\Laboratory\Transformers fault tollerance


## Global variables

In [2]:
RESULTS_DIR = ROOT / "results" / "base_transformer" / "ag_news"
MODEL_NAME = "base_transformer_agnews"
DATA_DIR = ROOT / "data" / "ag_news"
# NVIDIA GPU when one is visible; FT_DEVICE=cpu reproduces the CPU-only run
DEVICE = select_device()

MAX_LEN = C._env_int("FT_MAX_LEN", 64)
VOCAB_SIZE = 30_000
NUM_CLASSES = 4
CLASS_NAMES = ["World", "Sports", "Business", "Sci/Tech"]

MODEL_CONFIG = dict(C.BASE_PARAM, activation="relu")

TRAIN_CONFIG = {
    "loss": "CrossEntropyLoss",
    "optimizer": "AdamW",
    # EXP_A trains its synthetic task at 1e-4/1e-2; a from-scratch text encoder
    # needs a slightly larger step to converge inside a CPU epoch budget.
    "learning_rate": 3e-4,
    "weight_decay": 1e-2,
    "batch_size": 128,
    "epochs": C._env_int("FT_EPOCHS", 12),
    "patience": C._env_int("FT_PATIENCE", 3),
    "grad_clip": 1.0,
    "selection_metric": "roc_auc",
    "cosine_schedule": True,
}

# degradation variables (as in EXP_A)
CRIT_COL = "D_pos_mean"       # criticality signal
DEG_COL = "D_mean"            # degradation variable for the fault-severity analysis
SAFE_COL = "D_pos_mean"       # safe-region degradation variable
DEG_LABEL = "mean probability damage"
SAFE_LABEL = "mean positive damage"
TAUS = C.CLS_TAUS
TAU_LABELS = ["tau_%.3f" % t for t in TAUS]

print("device:", device_report(DEVICE))
print("MAX_LEN", MAX_LEN, "| epochs", TRAIN_CONFIG["epochs"])

device: cuda:0 NVIDIA GeForce RTX 4070 Laptop GPU | 8.0 GB | sm_89 | torch cuda 12.6 | tf32 off
MAX_LEN 64 | epochs 12


In [3]:
OUT = pathlib.Path(RESULTS_DIR)
TABLES, FIGURES, MODELS = OUT / "tables", OUT / "figures", OUT / "models"
for d in (TABLES, FIGURES, MODELS):
    d.mkdir(parents=True, exist_ok=True)

def save(df, name):
    path = TABLES / name
    df.to_csv(path, index=False)
    print("saved", path.relative_to(ROOT), df.shape)
    return path

print("artifacts ->", OUT)
print("K_TRIALS=%d  N_FAULT=%d" % (C.K_TRIALS, C.N_FAULT))

artifacts -> D:\Laboratory\Transformers fault tollerance\experiments\base_transformer\ag_news
K_TRIALS=10000  N_FAULT=1000


## STEP 1: AG News dataset

In [4]:
import collections, re

TOKEN_RE = re.compile(r"[a-z0-9']+")

def read_split(name):
    df = pd.read_csv(DATA_DIR / ("%s.csv" % name), header=None,
                     names=["label", "title", "description"])
    text = (df["title"].fillna("") + " " + df["description"].fillna("")).str.lower()
    return text.tolist(), (df["label"].to_numpy() - 1).astype(np.int64)

train_text, train_y = read_split("train")
test_text, test_y = read_split("test")
print("raw: train=%d test=%d classes=%d" % (len(train_text), len(test_text), len(np.unique(train_y))))
print("class balance:", np.bincount(train_y))
pd.DataFrame({"label": [CLASS_NAMES[i] for i in train_y[:5]], "text": train_text[:5]})

raw: train=120000 test=7600 classes=4
class balance: [30000 30000 30000 30000]


,label,text
0,Business,wall st. bears claw back into the black (reute...
1,Business,carlyle looks toward commercial aerospace (reu...
2,Business,oil and economy cloud stocks' outlook (reuters...
3,Business,iraq halts oil exports from main southern pipe...
4,Business,"oil prices soar to all-time record, posing new..."


In [5]:
def build_vocab(texts, vocab_size):
    counter = collections.Counter()
    for t in texts:
        counter.update(TOKEN_RE.findall(t))
    vocab = {"<pad>": 0, "<unk>": 1}
    for token, _ in counter.most_common(vocab_size - 2):
        vocab[token] = len(vocab)
    return vocab

def encode(texts, vocab, max_len):
    unk = vocab["<unk>"]
    out = np.zeros((len(texts), max_len), dtype=np.int64)
    for i, t in enumerate(texts):
        ids = [vocab.get(tok, unk) for tok in TOKEN_RE.findall(t)][:max_len]
        out[i, :len(ids)] = ids
    return torch.from_numpy(out)

vocab = build_vocab(train_text, VOCAB_SIZE)
X_all = encode(train_text, vocab, MAX_LEN)
X_test = encode(test_text, vocab, MAX_LEN)
print("vocab=%d  encoded train %s  test %s" % (len(vocab), tuple(X_all.shape), tuple(X_test.shape)))

vocab=30000  encoded train (120000, 64)  test (7600, 64)


In [6]:
def balanced_subset(y, n_total, seed):
    rng = np.random.default_rng(seed)
    classes = np.unique(y)
    per = n_total // len(classes)
    picks = [rng.choice(np.flatnonzero(y == c), size=min(per, int((y == c).sum())), replace=False)
             for c in classes]
    out = np.concatenate(picks)
    rng.shuffle(out)
    return out

rng = np.random.default_rng(C.SPLIT_SEED)
perm = rng.permutation(len(X_all))
n_val = 12_000
val_idx, train_idx = perm[:n_val], perm[n_val:]

n_sub = C._env_int("FT_TRAIN_SUBSAMPLE", 0)
if n_sub:
    train_idx = train_idx[:n_sub]

X_train, y_train = X_all[train_idx], train_y[train_idx]
X_val, y_val = X_all[val_idx], train_y[val_idx]

# the official test split plays the role of EXP_A's independent fault-analysis
# dataset: a calibration part for single-head criticality and a fixed balanced
# part reused by every random multi-head trial
eval_idx = balanced_subset(test_y, C.N_FAULT, C.FAULT_SPLIT_SEED)
calib_mask = np.ones(len(test_y), dtype=bool)
calib_mask[eval_idx] = False
X_calib, y_calib = X_test[calib_mask], test_y[calib_mask]
X_eval, y_eval = X_test[eval_idx].contiguous(), test_y[eval_idx]

print("train=%d val=%d calib=%d fault_eval=%d"
      % (len(X_train), len(X_val), len(X_calib), len(X_eval)))
print("fault-eval class balance:", np.bincount(y_eval))

train=108000 val=12000 calib=6600 fault_eval=1000
fault-eval class balance: [250 250 250 250]


## STEP 2: encoder-only Transformer classifier

In [7]:
from faulttol.metrics import (classification_clean_metrics, classification_damage,
                              classification_outputs)
from faulttol.model import TextClassifier, count_params
from faulttol.training import train_classifier

torch.manual_seed(C.TRAIN_SEED)
np.random.seed(C.TRAIN_SEED)

model = TextClassifier(len(vocab), MODEL_CONFIG, max_len=MAX_LEN,
                       num_classes=NUM_CLASSES, activation="relu")
print("params = %d" % count_params(model))
print("heads  = %d layers x %d = %d"
      % (MODEL_CONFIG["num_layers"], MODEL_CONFIG["num_heads"], len(get_all_heads(model))))
model

params = 10840324
heads  = 4 layers x 8 = 32


TextClassifier(
  (pos): PositionalEncoding()
  (layers): ModuleList(
    (0-3): 4 x EncoderLayer(
      (attn): MultiHeadAttention(
        (W_q): Linear(in_features=256, out_features=256, bias=True)
        (W_k): Linear(in_features=256, out_features=256, bias=True)
        (W_v): Linear(in_features=256, out_features=256, bias=True)
        (W_o): Linear(in_features=256, out_features=256, bias=True)
      )
      (ff): Sequential(
        (0): Linear(in_features=256, out_features=1024, bias=True)
        (1): ReLU()
        (2): Dropout(p=0.05, inplace=False)
        (3): Linear(in_features=1024, out_features=256, bias=True)
      )
      (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.05, inplace=False)
    )
  )
  (dropout): Dropout(p=0.05, inplace=False)
  (embedding): Embedding(30000, 256, padding_idx=0)
  (classifier): Linear(in_features=256, out

In [8]:
hist = train_classifier(
    model, X_train, y_train, X_val, y_val, DEVICE,
    lr=TRAIN_CONFIG["learning_rate"], weight_decay=TRAIN_CONFIG["weight_decay"],
    epochs=TRAIN_CONFIG["epochs"], batch_size=TRAIN_CONFIG["batch_size"],
    patience=TRAIN_CONFIG["patience"], grad_clip=TRAIN_CONFIG["grad_clip"],
    ckpt_path=MODELS / "best.pt",
    config={"model": MODEL_CONFIG, "train": TRAIN_CONFIG, "vocab_size": len(vocab)},
    selection_metric=TRAIN_CONFIG["selection_metric"],
    cosine=TRAIN_CONFIG["cosine_schedule"],
)
save(pd.DataFrame(hist["history"]), "training_history.csv")
pd.DataFrame(hist["history"])

    epoch  1  loss 0.4878  val_acc 0.8850  val_f1 0.8842  val_auc 0.9795  (36s)


    epoch  2  loss 0.2755  val_acc 0.9032  val_f1 0.9024  val_auc 0.9836  (36s)


    epoch  3  loss 0.2161  val_acc 0.9064  val_f1 0.9060  val_auc 0.9844  (36s)


    epoch  4  loss 0.1735  val_acc 0.9137  val_f1 0.9134  val_auc 0.9858  (36s)


    epoch  5  loss 0.1370  val_acc 0.9135  val_f1 0.9131  val_auc 0.9856  (36s)


    epoch  6  loss 0.1029  val_acc 0.9150  val_f1 0.9146  val_auc 0.9860  (36s)


... (2 outputs left out)

    epoch  9  loss 0.0370  val_acc 0.9143  val_f1 0.9139  val_auc 0.9856  (36s)


    early stopping at epoch 9


saved experiments\base_transformer\ag_news\tables\training_history.csv (9, 8)


,epoch,train_loss,secs,val_accuracy,val_precision,val_recall,val_f1,val_roc_auc
0,1,0.487754,35.744129,0.885000,0.886211,0.884868,0.884190,0.979463
1,2,0.275542,35.781545,0.903167,0.903024,0.902544,0.902404,0.983630
2,3,0.216081,35.888079,0.906417,0.908233,0.905586,0.906047,0.984407
3,4,0.173457,35.995417,0.913667,0.913990,0.913288,0.913437,0.985792
4,5,0.137031,36.132972,0.913500,0.913288,0.913037,0.913085,0.985649
5,6,0.102898,36.208611,0.915000,0.914604,0.914691,0.914584,0.985987
6,7,0.075446,36.282967,0.915417,0.915307,0.914874,0.914918,0.985452
7,8,0.054843,36.415359,0.916417,0.916299,0.916012,0.915988,0.985517
8,9,0.037038,36.364219,0.914333,0.914040,0.913981,0.913885,0.985601


In [9]:
calib_metrics = classification_clean_metrics(
    classification_outputs(model, X_calib, y_calib, DEVICE))

def evaluate():
    return classification_outputs(model, X_eval, y_eval, DEVICE, chunk=1000)

damage_fn = classification_damage
clean_eval = classification_clean_metrics(evaluate())

t0 = time.time()
for _ in range(5):
    evaluate()
per_eval = (time.time() - t0) / 5.0

print("calibration set (n=%d):" % len(X_calib), calib_metrics)
print("fixed eval subset (n=%d):" % len(X_eval), clean_eval)
print("cost: %.3f s per configuration -> %.1f min for K=%d"
      % (per_eval, per_eval * C.K_TRIALS / 60.0, C.K_TRIALS))

attn_stats = collect_attention_stats_per_head(
    model, make_loader(X_eval, torch.as_tensor(y_eval), 250, shuffle=False), DEVICE)
save(pd.DataFrame(attn_stats), "attention_stats.csv")

EXTRA_META = {
    "dataset": "AG News", "class_names": CLASS_NAMES, "max_len": MAX_LEN,
    "vocab_size": len(vocab), "train_config": TRAIN_CONFIG,
    "model_config": MODEL_CONFIG, "calibration_clean_metrics": calib_metrics,
    "n_params": count_params(model), "best_val_score": hist["best_score"],
    "seconds_per_evaluation": per_eval,
}

calibration set (n=6600): {'accuracy': 0.9118181818181819, 'precision': 0.9117736100986529, 'recall': 0.9118181818181817, 'f1': 0.9117583631437441, 'roc_auc': 0.9850493878175697}
fixed eval subset (n=1000): {'accuracy': 0.907, 'precision': 0.9070246341055538, 'recall': 0.9069999999999999, 'f1': 0.906797791029441, 'roc_auc': 0.9845946666666667}
cost: 0.102 s per configuration -> 17.0 min for K=10000
saved experiments\base_transformer\ag_news\tables\attention_stats.csv (32, 5)


## Fault model

**Definition 1 (attention-head crash).** A head is crashed when its output tensor is
replaced with zeros *after* the attention weights multiply V and *before* head
concatenation and the output projection `W_o`.

Faults are injected **at inference only** - the model is trained clean, then frozen.
After every faulted evaluation all masks are restored to 1.

## STEP 3: single-head crash

Every head is crashed in turn while all others stay alive, so each clean/fault comparison differs in exactly one head.

In [10]:
sh_df = single_head.run_single_head_analysis(
    model=model,
    evaluate=evaluate,
    damage_fn=damage_fn,
    criticality_col=CRIT_COL,
    model_name=MODEL_NAME,
    verbose=True,
)
save(sh_df, "single_head_damage.csv")
sh_df.nlargest(10, "criticality")[["head_name", "criticality", "redundancy", DEG_COL]]

  L0_H0    D_pos_mean = 0.008216


  L0_H1    D_pos_mean = 0.006681


  L0_H2    D_pos_mean = 0.005102


  L0_H3    D_pos_mean = 0.007888


  L0_H4    D_pos_mean = 0.012335


  L0_H5    D_pos_mean = 0.007712


... (24 outputs left out)

  L3_H6    D_pos_mean = 0.010269


  L3_H7    D_pos_mean = 0.009770


saved experiments\base_transformer\ag_news\tables\single_head_damage.csv (32, 30)


,head_name,criticality,redundancy,D_mean
4,L0_H4,0.012335,8.107237e-11,0.008116
12,L1_H4,0.011205,9.160279e-02,0.003852
10,L1_H2,0.010637,1.376200e-01,0.006600
30,L3_H6,0.010269,1.674343e-01,0.003361
31,L3_H7,0.009770,2.078902e-01,0.001227
18,L2_H2,0.009633,2.190563e-01,0.002639
19,L2_H3,0.009409,2.372096e-01,0.003541
20,L2_H4,0.009383,2.393319e-01,0.001184
24,L3_H0,0.009298,2.462055e-01,0.002501
22,L2_H6,0.009185,2.553609e-01,0.004403


### Layer fragility and vulnerability map

In [11]:
layer_summary = single_head.build_layer_summary(sh_df)
model_summary = single_head.build_model_summary(sh_df, layer_summary, MODEL_NAME, clean_eval)
save(layer_summary, "layer_summary.csv")

plots.vulnerability_map(sh_df, "%s - single-head vulnerability (%s)" % (MODEL_NAME, CRIT_COL),
                        FIGURES / "vulnerability_map.png")
plots.criticality_profile(sh_df, "%s - criticality concentration" % MODEL_NAME,
                          FIGURES / "criticality_profile.png", ylabel=CRIT_COL)

print("most critical head      :", model_summary["most_critical_head"],
      "(criticality %.6f)" % model_summary["max_criticality"])
print("median / max criticality: %.4f" % (model_summary["median_criticality"] / model_summary["max_criticality"]))
print("mean redundancy         : %.4f" % model_summary["mean_redundancy"])
print("fragile layer by mean   :", model_summary["most_fragile_layer_by_mean"])
print("fragile layer by max    :", model_summary["most_fragile_layer_by_max"])
layer_summary

saved experiments\base_transformer\ag_news\tables\layer_summary.csv (4, 7)


most critical head      : L0_H4 (criticality 0.012335)
median / max criticality: 0.6730
mean redundancy         : 0.3274
fragile layer by mean   : 1
fragile layer by max    : 0


,layer,mean_layer_criticality,max_head_criticality,min_head_criticality,mean_redundancy,n_heads,most_critical_head
0,0,0.007769,0.012335,0.005102,0.370114,8,L0_H4
1,1,0.008758,0.011205,0.005346,0.289951,8,L1_H4
2,2,0.008503,0.009633,0.005679,0.310647,8,L2_H2
3,3,0.008155,0.010269,0.006479,0.338856,8,L3_H6


## STEP 4: criticality, redundancy and severity weights

`C = mean(max(damage,0))`, `R = 1 - C/max(C)`, `W = C/mean(C)`. Critical heads are the top 10% by criticality.

In [12]:
weight_lookup = single_head.build_head_weight_lookup(sh_df)
critical_heads = single_head.critical_head_set(sh_df, C.CRITICAL_TOP_FRACTION)
print("critical heads (top %d%%): %s"
      % (100 * C.CRITICAL_TOP_FRACTION, ["L%d_H%d" % h for h in critical_heads]))
print("heads with near-zero effect:", int((sh_df["criticality"] <= 1e-6).sum()), "of", len(sh_df))
save(sh_df[["head_name", "layer", "head", "criticality", "redundancy", "severity_weight"]],
     "head_weights.csv")
sh_df.nsmallest(5, "criticality")[["head_name", "criticality", "redundancy", DEG_COL]]

critical heads (top 10%): ['L0_H4', 'L1_H4', 'L1_H2']
heads with near-zero effect: 0 of 32
saved experiments\base_transformer\ag_news\tables\head_weights.csv (32, 6)


,head_name,criticality,redundancy,D_mean
2,L0_H2,0.005102,0.586401,0.000416
13,L1_H5,0.005346,0.566561,0.000013
7,L0_H7,0.005616,0.544691,0.000995
16,L2_H0,0.005679,0.539595,0.001009
26,L3_H2,0.006479,0.474735,-0.000370


## STEP 5: random multi-head fault injection

`N_fail ~ Uniform{1..32}`, heads drawn without replacement and crashed simultaneously, evaluated on the fixed subset.

In [13]:
n_heads = len(get_all_heads(model))
mh_df = multihead.run_random_multihead(
    model=model,
    evaluate=evaluate,
    damage_fn=damage_fn,
    weight_lookup=weight_lookup,
    critical_heads=critical_heads,
    model_name=MODEL_NAME,
    k_trials=C.K_TRIALS,
    seed=C.MULTIHEAD_FAULT_SEED,
    num_layers=model.num_layers,
)
save(mh_df, "random_multihead.csv")
mh_df[["N_fail", "N_crit_fail", "W_fail", "W_fail_mean", DEG_COL]].describe()

    trial 500/10000  9.2 trials/s  eta 17.3 min


    trial 1000/10000  9.2 trials/s  eta 16.3 min


    trial 1500/10000  9.2 trials/s  eta 15.4 min


    trial 2000/10000  9.2 trials/s  eta 14.5 min


    trial 2500/10000  9.2 trials/s  eta 13.6 min


    trial 3000/10000  9.2 trials/s  eta 12.7 min


... (12 outputs left out)

    trial 9500/10000  9.2 trials/s  eta 0.9 min


    trial 10000/10000  9.2 trials/s  eta 0.0 min


saved experiments\base_transformer\ag_news\tables\random_multihead.csv (10000, 36)


,N_fail,N_crit_fail,W_fail,W_fail_mean,D_mean
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,16.575600,1.560700,16.571797,1.000490,0.145612
std,9.164163,1.087857,9.166071,0.063163,0.167668
min,1.000000,0.000000,0.614918,0.614918,-0.000370
25%,9.000000,1.000000,8.808911,0.979065,0.024874
50%,17.000000,2.000000,16.579355,1.000000,0.073757
75%,24.000000,3.000000,24.429499,1.021012,0.206998
max,32.000000,3.000000,32.000000,1.486750,0.639192


### Degradation by fault fraction and failed-head criticality

In [14]:
grid = multihead.degradation_grid(mh_df, DEG_COL)
save(grid.reset_index(), "degradation_grid.csv")
plots.degradation_grid_plot(
    grid, "%s - mean %s by fault fraction and criticality" % (MODEL_NAME, DEG_LABEL),
    FIGURES / "degradation_grid.png", value_label=DEG_LABEL)
grid

saved experiments\base_transformer\ag_news\tables\degradation_grid.csv (5, 5)


crit_quartile,Q1,Q2,Q3,Q4
fault_bin,,,,
0-10%,0.002871,0.004139,0.003413,0.006791
10-25%,0.012289,0.015497,0.016184,0.019111
25-50%,0.038003,0.045261,0.048722,0.049582
50-75%,0.110578,0.131619,0.135190,0.136032
75-100%,0.273545,0.385328,0.432160,0.320176


### How well does fault severity explain degradation?

Rank correlations between each fault-severity descriptor and the degradation, plus the `W_fail` / `N_fail` confound.

In [15]:
from scipy.stats import kendalltau, pearsonr, spearmanr

def severity_correlations(df, deg_col):
    rows = []
    for sev in ("N_fail", "L_fault", "N_crit_fail", "W_fail", "W_fail_mean"):
        x, y = df[sev].to_numpy(float), df[deg_col].to_numpy(float)
        pr, pp = pearsonr(x, y)
        sr, sp = spearmanr(x, y)
        kr, kp = kendalltau(x, y)
        rows.append({"severity": sev, "degradation": deg_col, "pearson": pr,
                     "spearman": sr, "kendall": kr, "spearman_p": sp})
    return pd.DataFrame(rows)

corr_df = severity_correlations(mh_df, DEG_COL)
save(corr_df, "severity_correlations.csv")

# W_fail is partly a restatement of N_fail - report the confound explicitly
conf = spearmanr(mh_df["W_fail"], mh_df["N_fail"]).statistic
print("Spearman(W_fail, N_fail) = %.3f  <- how much of W_fail is just fault count" % conf)
corr_df

saved experiments\base_transformer\ag_news\tables\severity_correlations.csv (5, 6)
Spearman(W_fail, N_fail) = 0.999  <- how much of W_fail is just fault count


,severity,degradation,pearson,spearman,kendall,spearman_p
0,N_fail,D_mean,0.866405,0.992010,0.932775,0.000000e+00
1,L_fault,D_mean,0.355872,0.650324,0.536355,0.000000e+00
2,N_crit_fail,D_mean,0.697613,0.807804,0.664020,0.000000e+00
3,W_fail,D_mean,0.868176,0.994690,0.935495,0.000000e+00
4,W_fail_mean,D_mean,0.011850,0.061877,0.040668,5.908043e-10


## STEP 6: empirical probabilistic fault-tolerance regions

Widest `(n, c)` with `P(Y <= tau | N_fail <= n, N_crit_fail <= c) >= 0.95`.

In [16]:
grids, safe_summary = saferegion.run_safe_region_analysis(
    mh_df, SAFE_COL, TAUS, n_heads, len(critical_heads),
    min_configs=C.MIN_CONFIGS_PER_BOUNDARY, safe_prob=C.SAFE_PROB,
    tau_labels=TAU_LABELS)
save(grids, "safe_region_grid.csv")
save(safe_summary, "safe_region_summary.csv")

tau_df = saferegion.required_tau_table(
    mh_df, SAFE_COL, n_heads, len(critical_heads),
    min_configs=C.MIN_CONFIGS_PER_BOUNDARY, safe_prob=C.SAFE_PROB)
save(tau_df, "required_tau.csv")
plots.required_tau_plot(
    tau_df, "%s - required degradation threshold for 95%% safety" % MODEL_NAME,
    FIGURES / "required_tau.png", ylabel="tau required (%s)" % SAFE_LABEL)

safe_summary

saved experiments\base_transformer\ag_news\tables\safe_region_grid.csv (384, 9)
saved experiments\base_transformer\ag_news\tables\safe_region_summary.csv (3, 9)
saved experiments\base_transformer\ag_news\tables\required_tau.csv (128, 6)


,tau,safe_boundary,n_fail_max,n_crit_fail_max,max_p_safe,coverage_percent,smallest_required_tau,n_boundaries_evaluated,threshold_name
0,0.010,"N_fail<=1, N_crit_fail<=0",1,0,0.977186,2.63,0.00977,128,tau_0.010
1,0.025,"N_fail<=4, N_crit_fail<=3",4,3,0.959283,12.28,0.00977,128,tau_0.025
2,0.050,"N_fail<=10, N_crit_fail<=3",10,3,0.956622,30.43,0.00977,128,tau_0.050


## STEP 7: summary

In [17]:
summary = {
    "model_name": MODEL_NAME,
    "n_heads": n_heads,
    "n_layers": model.num_layers,
    "k_trials": int(C.K_TRIALS),
    "n_fault_eval_samples": int(C.N_FAULT),
    "clean_metrics": clean_eval,
    "single_head": model_summary,
    "critical_heads": ["L%d_H%d" % h for h in critical_heads],
    "criticality_col": CRIT_COL,
    "degradation_col": DEG_COL,
    "safe_degradation_col": SAFE_COL,
    "severity_correlations": corr_df.to_dict(orient="records"),
    "safe_regions": safe_summary.to_dict(orient="records"),
    "criticality_profile": sorted(sh_df["criticality"].tolist(), reverse=True),
    "median_over_max_criticality": float(sh_df["criticality"].median() / (sh_df["criticality"].max() + 1e-12)),
    "n_heads_zero_effect": int((sh_df["criticality"] <= 1e-6).sum()),
    "worst_single_head": {
        "head": sh_df.loc[sh_df["criticality"].idxmax(), "head_name"],
        "criticality": float(sh_df["criticality"].max()),
        "degradation": float(sh_df.loc[sh_df["criticality"].idxmax(), DEG_COL]),
        "safe_degradation": float(sh_df.loc[sh_df["criticality"].idxmax(), SAFE_COL]),
    },
    **EXTRA_META,
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
print("wrote", (OUT / "summary.json").relative_to(ROOT))

print()
print("=" * 70)
print("CONCLUSIONS -", MODEL_NAME)
print("=" * 70)
print("clean performance      :", clean_eval)
print("worst single head      : %s  ->  %s = %.6f"
      % (summary["worst_single_head"]["head"], DEG_COL, summary["worst_single_head"]["degradation"]))
print("median/max criticality : %.4f  (1.0 = uniform, 0.0 = fully concentrated)"
      % summary["median_over_max_criticality"])
print("critical heads         :", summary["critical_heads"])
print("layer of critical heads:", sorted({h.split("_")[0] for h in summary["critical_heads"]}))
for r in summary["safe_regions"]:
    print("safe region tau=%-10s: %s (P=%.3f, coverage %.2f%%)"
          % (r.get("threshold_name"), r.get("safe_boundary"), r.get("max_p_safe") or float("nan"),
             r.get("coverage_percent") or 0.0))

wrote experiments\base_transformer\ag_news\summary.json

CONCLUSIONS - base_transformer_agnews
clean performance      : {'accuracy': 0.907, 'precision': 0.9070246341055538, 'recall': 0.9069999999999999, 'f1': 0.906797791029441, 'roc_auc': 0.9845946666666667}
worst single head      : L0_H4  ->  D_mean = 0.008116
median/max criticality : 0.6730  (1.0 = uniform, 0.0 = fully concentrated)
critical heads         : ['L0_H4', 'L1_H4', 'L1_H2']
layer of critical heads: ['L0', 'L1']
safe region tau=tau_0.010 : N_fail<=1, N_crit_fail<=0 (P=0.977, coverage 2.63%)
safe region tau=tau_0.025 : N_fail<=4, N_crit_fail<=3 (P=0.959, coverage 12.28%)
safe region tau=tau_0.050 : N_fail<=10, N_crit_fail<=3 (P=0.957, coverage 30.43%)
